In [3]:
import os
if not os.path.exists('/content/KM3Former'):
    !git clone -b muon-count https://github.com/High-Energy-Physics-Institute/KM3Former.git KM3Former
%cd /content/KM3Former

Cloning into 'KM3Former'...
remote: Enumerating objects: 137, done.
remote: Counting objects: 100% (137/137), done.
remote: Compressing objects: 100% (76/76), done.
remote: Total 137 (delta 66), reused 119 (delta 52), pack-reused 0 (from 0)
Receiving objects: 100% (137/137), 180.31 KiB | 6.44 MiB/s, done.
Resolving deltas: 100% (66/66), done.
/content/KM3Former


In [4]:
!uv pip install km3io awkward tqdm pandas joblib --system

import sys
import os
repo_path = '/content/KM3Former'
if repo_path not in sys.path:
    sys.path.append(repo_path)

Using Python 3.12.13 environment at: /usr
Checked 5 packages in 91ms


In [5]:
import torch
!nvidia-smi
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

Fri Mar 27 10:21:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [6]:
!python pre-porcessing/pre_process.py \
  --source-format hdf5 \
  --task muon_count \
  --data-path ./data \
  --h5-path ./data/muon_data_7224_7247.h5 \
  --h5-hits-dataset hits \
  --h5-label-dataset mc_muons \
  --h5-label-col 0 \
  --count-class-values 0,1,2

In [9]:
%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True

env: PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True


In [10]:
!python model/train.py --config configs/train.muon_count.colab.json


Training:  19% 413/2156 [00:15<01:00, 28.89it/s]
Training:  19% 416/2156 [00:15<00:59, 29.02it/s]
Training:  19% 419/2156 [00:15<00:59, 29.08it/s]
Training:  20% 422/2156 [00:15<01:00, 28.81it/s]
Training:  20% 426/2156 [00:15<00:58, 29.55it/s]
Training:  20% 430/2156 [00:15<00:57, 29.78it/s]
Training:  20% 433/2156 [00:15<01:01, 28.08it/s]
Training:  20% 436/2156 [00:15<01:00, 28.40it/s]
Training:  20% 439/2156 [00:16<01:00, 28.33it/s]
Training:  21% 442/2156 [00:16<00:59, 28.69it/s]
Training:  21% 445/2156 [00:16<01:00, 28.51it/s]
Training:  21% 448/2156 [00:16<00:59, 28.90it/s]
Training:  21% 451/2156 [00:16<00:59, 28.66it/s]
Training:  21% 455/2156 [00:16<00:58, 29.29it/s]
Training:  21% 458/2156 [00:16<00:58, 29.02it/s]
Training:  21% 461/2156 [00:16<00:59, 28.52it/s]
Training:  22% 465/2156 [00:16<00:59, 28.65it/s]
Training:  22% 468/2156 [00:17<00:58, 28.97it/s]
Training:  22% 472/2156 [00:17<00:57, 29.45it/s]
Training:  22% 475/2156 [00:17<00:57, 29.06it/s]
Training:  22% 478/

In [11]:
!python model/infer.py \
  --checkpoint-path ./runs/muon_count_colab/best_model.pth \
  --split test \
  --data-path ./data \
  --output-path ./runs/muon_count_colab/test_predictions.pt

In [12]:
import torch
p=torch.load('runs/muon_count_colab/test_predictions.pt', map_location='cpu')
print({k:(tuple(v.shape) if hasattr(v,'shape') else v) for k,v in p.items() if k in ['task','target_kind','predictions','probabilities','predicted_classes','predicted_labels']})


{'task': 'muon_count', 'target_kind': 'multiclass', 'predictions': (2156, 3), 'probabilities': (2156, 3), 'predicted_classes': (2156,), 'predicted_labels': (2156,)}


In [13]:
y=torch.load('data/test_targets.pt', map_location='cpu')
pred=p['predicted_classes']
print({'test_accuracy': float((pred==y).float().mean())})


{'test_accuracy': 0.5329313278198242}


In [14]:
!zip -r /content/KM3Former/runs/muon_count_colab.zip /content/KM3Former/runs/muon_count_colab

  adding: content/KM3Former/runs/muon_count_colab/ (stored 0%)
  adding: content/KM3Former/runs/muon_count_colab/model_epoch_6.pth (deflated 11%)
  adding: content/KM3Former/runs/muon_count_colab/model_epoch_9.pth (deflated 11%)
  adding: content/KM3Former/runs/muon_count_colab/model_epoch_5.pth (deflated 11%)
  adding: content/KM3Former/runs/muon_count_colab/test_predictions.pt (deflated 43%)
  adding: content/KM3Former/runs/muon_count_colab/model_epoch_1.pth (deflated 11%)
  adding: content/KM3Former/runs/muon_count_colab/model_epoch_2.pth (deflated 11%)
  adding: content/KM3Former/runs/muon_count_colab/model_epoch_7.pth (deflated 11%)
  adding: content/KM3Former/runs/muon_count_colab/tensorboard/ (stored 0%)
  adding: content/KM3Former/runs/muon_count_colab/tensorboard/events.out.tfevents.1774607070.63af4cfefe29.14618.0 (deflated 9%)
  adding: content/KM3Former/runs/muon_count_colab/tensorboard/1774608052.5936568/ (stored 0%)
  adding: content/KM3Former/runs/muon_count_colab/tensorb